# Cheap + light derivations (companion to `radiomap_6g_sweep.ipynb`)

Fills the **inexpensive** entries of `derived.json` so the main notebook's orchestrator (cell 119) only has the heavy ones left. The main orchestrator skips anything already derived (`have()`), so results from here are picked up on resume.

| Step | Parameters | Cost |
|---|---|---|
| A | `huber_sigma_db` (4 scenarios x 3 densities) | seconds (MAD of residuals) |
| B | `kmeans_n_init` | about a minute (KMeans) |
| C | the 21 **BOUNDARY** `idw_power_*` / `krig_nlags_*` entries, grids widened, flagged cells re-run | minutes (LOO / K-fold kriging, no NN training) |
| D | `spatial_folds`, `spatial_buffer_m` | seconds (needs B and C first) |
| E | the 11 missing `knn_k` searches | **the costly one here**: about 175 small GCN fits per (scenario, density) -> about 1,900 fits |
| F | *(optional, off)* `val_scheme`, `val_frac`, GCN/GAT `patience`, `epochs_cap`, `grad_clip` | provisional only, see the note in F |

**Before running**
- Put this file next to `radiomap_6g_sweep.ipynb` and `derived.json` (no `derived.complete`).
- Needs the same environment as the main notebook (torch, torch_geometric, deepmimo, pykrige, ...) and internet access to `deepmimo.net` (scenario discovery and loading).
- Step 1 re-executes the main notebook's definition cells (all `%%report` cells and the final orchestrator trigger are skipped). It also runs scenario discovery and loads the scenarios, so it is not instant.
- `derived.json` is backed up first; every step saves as it goes and is safe to re-run.

In [1]:
import os, json, shutil, time, math, contextlib
from datetime import datetime

MAIN_NB = "radiomap_6g_sweep.ipynb"     # path to the main notebook
WORKDIR = "."                           # folder that holds derived.json
RUN_DEPENDENT = False                   # step F (provisional), see its markdown first

os.chdir(WORKDIR)
os.environ.setdefault("RM_WORKERS", "6")   # parallel forked workers (~1 GB each). Use 1 on Windows (no fork) or low RAM.

assert os.path.exists("derived.json"), "derived.json not found in WORKDIR"
assert not os.path.exists("derived.complete"), "derived.complete exists -> main notebook is in pass 2; delete it to derive"
bak = f"derived.backup.{datetime.now():%Y%m%d_%H%M%S}.json"
shutil.copy("derived.json", bak); print("backup ->", bak)

@contextlib.contextmanager
def timed(label):
    t = time.time(); print(f"--- {label} ..."); yield
    print(f"--- {label}: {time.time() - t:.0f}s")

backup -> derived.backup.20261003_235946.json


## 1. Load the main notebook's definitions (no sweeps, no orchestrator)

In [2]:
nbk = json.load(open(MAIN_NB))
cells = nbk["cells"]
last = next(i for i, c in enumerate(cells) if c["cell_type"] == "code" and "def run_derivation" in "".join(c["source"]))
SKIP = set()    # optionally add cell indices to skip, e.g. {18} to skip the catalog listing

with timed("loading definitions"):
    for i, c in enumerate(cells[: last + 1]):
        if c["cell_type"] != "code" or i in SKIP: continue
        src = "".join(c["source"])
        if src.lstrip().startswith("%%report"): continue
        if i == last:                                   # keep the defs, drop the trailing `if PASS == 1: run_derivation(...)`
            src = src[: src.index("\nif PASS == 1:\n    run_derivation(")]
        res = get_ipython().run_cell(src, store_history=False)
        res.raise_error()
assert PASS == 1, "main notebook thinks it is in pass 2"
print("loaded cells 0..%d; PASS=%d" % (last, PASS))

--- loading definitions ...
/home/praaneshnair/gitProjects/prism/.venv/bin/python3
2026-10-03 23:59:46.311864


/home/praaneshnair/gitProjects/prism/.venv/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


python                          : 3.12.12 (main, Nov 19 2025, 22:46:53) [Clang 21.1.4 ]
platform                        : Linux-7.0.13-arch1-2-x86_64-with-glibc2.43
executable                      : /home/praaneshnair/gitProjects/prism/.venv/bin/python3
torch                           : 2.14.0+cu130
torch_geometric                 : 2.8.0.post1
numpy                           : 2.2.6
scipy                           : 1.18.1
sklearn                         : 1.9.1
pandas                          : 3.0.5
matplotlib                      : 3.11.2
pykrige                         : 1.7.3
deepmimo                        : 4.0.5
optuna                          : 5.0.0
tabulate                        : 0.10.0
cuda_available                  : False
device_used_in_this_notebook    : cpu
Audited 10 packages in 39ms
[pass 1] derivation stage
[registry] loaded derived.json
NUM_SEEDS = 20  (source: assumed)
i1_2p5
i2_28b


['i1_2p5', 'i2_28b']

Full catalog: 201 scenarios
Download failed: 403 Client Error: Forbidden for url: https://deepmimo.net/api/download/secure?filename=i3_2p5.zip
Download failed: 403 Client Error: Forbidden for url: https://deepmimo.net/api/download/secure?filename=i2_28.zip
  candidate i1_2p4           <- indoor slug pattern i<k>_...
  candidate i1_2p5           <- tag environment=indoor
  candidate i2_28            <- probe slug: direct download succeeded
  candidate i2_28b           <- tag environment=indoor
  candidate i3_2p4           <- indoor slug pattern i<k>_...
  candidate i3_2p5           <- probe slug: direct download succeeded
  candidate i3_60            <- indoor slug pattern i<k>_...
Scenario "i1_2p4" already exists in /home/praaneshnair/gitProjects/prism/deepmimo_scenarios
Loading TXRX PAIR: TXset 0 (tx_idx 0) & RXset 1 (rx_idxs 5000)
Loading TXRX PAIR: TXset 0 (tx_idx 1) & RXset 1 (rx_idxs 5000)
Loading TXRX PAIR: TXset 0 (tx_idx 2) & RXset 1 (rx_idxs 5000)
Loading TXRX PAIR: TXset 0 (t

...

Generating channels: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 16/16 [00:00<00:00, 975.29it/s]


,scenario,layout,freq_ghz,n_points,extent_m,los_frac,rss_min,rss_max,rss_std,ok,indoor_scale,note
0,i1_2p4,i1,2.4,5000,4.0,1.0,-61.77,-51.23,1.58,True,True,
1,i1_2p5,i1,2.5,5000,4.0,1.0,-61.92,-51.49,1.69,True,True,
2,i2_28,i2,NaN,0,NaN,NaN,NaN,NaN,NaN,False,False,FileNotFoundError: Scenario folder not found: ...
3,i2_28b,i2,28.0,4834,7.0,0.0,-91.97,-73.05,3.69,True,True,
4,i3_2p4,i3,2.4,5000,5.5,1.0,-60.62,-47.96,1.57,True,True,
5,i3_2p5,i3,NaN,0,NaN,NaN,NaN,NaN,NaN,False,False,FileNotFoundError: Scenario folder not found: ...
6,i3_60,i3,60.0,5000,5.5,1.0,-87.99,-76.13,1.24,True,True,



Usable indoor scenarios: ['i2_28b', 'i1_2p5', 'i1_2p4', 'i3_2p4', 'i3_60']
Distinct layout families: 3 | distinct carrier bands (GHz): [np.float64(2.4), np.float64(2.5), np.float64(28.0), np.float64(60.0)]
Ordered train->test pairs by shift type: {'layout+band': 12, 'identical': 2, 'layout-only': 4, 'band-only': 2}

[RESULT] 3 distinct layouts available. Section 6f / 7c / 8c / 8d extend every sweep to them.
[resume] discovery also found ['i1_2p4']; ignored so derived.json stays consistent (delete scn_list from derived.json to include them)
Scenario "i2_28b" already exists in /home/praaneshnair/gitProjects/prism/deepmimo_scenarios
Loading TXRX PAIR: TXset 1 (tx_idx 0) & RXset 0 (rx_idxs 140901)


Generating channels: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  1.90it/s]


[load_scenario] i2_28b tx=0: dropped 4750/140901 no-path / sub-noise-floor points
(136151, 3) (136151,) (136151, 9) 28.0
--- loading definitions: 26s
loaded cells 0..119; PASS=1


## 2. Rebuild the pools exactly as the orchestrator does

In [3]:
FROZEN = D("scn_list")
if list(SCN_LIST) != list(FROZEN):
    print("WARNING: discovery returned", SCN_LIST, "but the frozen scn_list is", FROZEN, "-> using the frozen one")
    SCN_LIST = list(FROZEN)

with timed("build_data"):
    build_data()
verify_pool_identity()                      # pools must match the saved fingerprints, otherwise derived values would leak
print("pool identity OK for", SCN_LIST)

DENS_ALL = [d for d in _grid if d <= POOL_N // 3]
DENS_T = [DENS_ALL[0], DENS_ALL[len(DENS_ALL) // 2], DENS_ALL[-1]]     # tuning densities, as in run_derivation
DENS_D = DENS_T
print("DENS_T =", DENS_T, "| derivation seeds =", SEEDS_D, "| workers =", N_WORKERS)

def status():
    df = derivation_report()
    a = df[df.source.astype(str).str.startswith("assumed")]
    b = df[df.source.astype(str).str.contains("BOUNDARY")]
    print(f"entries={len(df)}  still assumed={len(a)} ({sorted(a.name.unique())})")
    print(f"BOUNDARY flags={len(b)}")
status()

--- build_data ...
Scenario "i2_28b" already exists in /home/praaneshnair/gitProjects/prism/deepmimo_scenarios
Loading TXRX PAIR: TXset 1 (tx_idx 0) & RXset 0 (rx_idxs 140901)


Generating channels: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  1.77it/s]


[load_scenario] i2_28b tx=0: dropped 4750/140901 no-path / sub-noise-floor points
Scenario "i1_2p5" already exists in /home/praaneshnair/gitProjects/prism/deepmimo_scenarios
Loading TXRX PAIR: TXset 2 (tx_idx 0) & RXset 0 (rx_idxs 5000)
Loading TXRX PAIR: TXset 2 (tx_idx 1) & RXset 0 (rx_idxs 5000)
Loading TXRX PAIR: TXset 2 (tx_idx 2) & RXset 0 (rx_idxs 5000)
Loading TXRX PAIR: TXset 2 (tx_idx 3) & RXset 0 (rx_idxs 5000)
Loading TXRX PAIR: TXset 2 (tx_idx 4) & RXset 0 (rx_idxs 5000)
Loading TXRX PAIR: TXset 2 (tx_idx 5) & RXset 0 (rx_idxs 5000)
Loading TXRX PAIR: TXset 2 (tx_idx 6) & RXset 0 (rx_idxs 5000)
Loading TXRX PAIR: TXset 2 (tx_idx 7) & RXset 0 (rx_idxs 5000)
Loading TXRX PAIR: TXset 2 (tx_idx 8) & RXset 0 (rx_idxs 5000)
Loading TXRX PAIR: TXset 2 (tx_idx 9) & RXset 0 (rx_idxs 5000)
Loading TXRX PAIR: TXset 2 (tx_idx 10) & RXset 0 (rx_idxs 5000)
Loading TXRX PAIR: TXset 2 (tx_idx 11) & RXset 0 (rx_idxs 5000)
Loading TXRX PAIR: TXset 2 (tx_idx 12) & RXset 0 (rx_idxs 5000)
Load

Generating channels: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 16/16 [00:00<00:00, 1310.52it/s]


Scenario "i3_2p4" already exists in /home/praaneshnair/gitProjects/prism/deepmimo_scenarios
Loading TXRX PAIR: TXset 2 (tx_idx 0) & RXset 0 (rx_idxs 10000)
Loading TXRX PAIR: TXset 3 (tx_idx 0) & RXset 0 (rx_idxs 10000)


Generating channels: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 16/16 [00:00<00:00, 520.77it/s]


Scenario "i3_60" already exists in /home/praaneshnair/gitProjects/prism/deepmimo_scenarios
Loading TXRX PAIR: TXset 2 (tx_idx 0) & RXset 0 (rx_idxs 66671)
Loading TXRX PAIR: TXset 3 (tx_idx 0) & RXset 0 (rx_idxs 66671)


Generating channels: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 16/16 [00:00<00:00, 92.89it/s]

--- build_data: 9s
pool identity OK for ['i2_28b', 'i1_2p5', 'i3_2p4', 'i3_60']
DENS_T = [20, 107, 327] | derivation seeds = (1, 2, 3, 4, 5) | workers = 6
entries=226  still assumed=41 (['dropout', 'epochs_cap', 'grad_clip', 'heads', 'hidden', 'huber_beta_db', 'huber_sigma_db', 'idw_nn_detrended', 'idw_nn_raw', 'idw_power_detrended', 'idw_power_raw', 'kernel_bw_m', 'kmeans_n_init', 'knn_k', 'krig_model_detrended', 'krig_model_raw', 'krig_nlags_detrended', 'krig_nlags_raw', 'krig_range', 'layers', 'lr', 'max_points', 'noise_floor_dbm', 'nopath_policy', 'num_seeds', 'patience', 'power_definition', 'spatial_buffer_m', 'spatial_folds', 'star_K', 'tune_exclude_n', 'unit_offset_db', 'val_frac', 'val_scheme', 'weight_decay'])
BOUNDARY flags=21


## A. `huber_sigma_db` (robust scale of detrended pool residuals)

In [4]:
with timed("huber_sigma_db"):
    derive_huber_sigma(); save_derived()
print({(k[1], k[2]): round(v["value"], 2) for k, v in DERIVED.items() if k[0] == "huber_sigma_db" and k[1]})

--- huber_sigma_db ...
--- huber_sigma_db: 0s
{('i2_28b', 20): 5.08, ('i2_28b', 107): 5.53, ('i2_28b', 327): 5.86, ('i1_2p5', 20): 1.49, ('i1_2p5', 107): 1.07, ('i1_2p5', 327): 1.15, ('i3_2p4', 20): 2.11, ('i3_2p4', 107): 2.02, ('i3_2p4', 327): 2.06, ('i3_60', 20): 0.89, ('i3_60', 107): 0.9, ('i3_60', 327): 0.92}


## B. `kmeans_n_init` (inertia check, 4 candidates x 5 seeds)

In [5]:
with timed("kmeans_n_init"):
    derive_kmeans_n_init(); save_derived()
print(DERIVED[("kmeans_n_init", None, None, None)]["source"])

--- kmeans_n_init ...
--- kmeans_n_init: 375s
SENSITIVE (inertia {1: 947.1, 3: 945.4, 10: 943.6, 30: 943.0})


## C. Re-run the BOUNDARY IDW / kriging searches on widened grids

`final_checks()` refuses to finish while any winner sits on a grid edge. The grids are widened as supersets, and the search is deterministic (same seeds, same folds), so entries whose optimum was interior stay optimal and **only the flagged (scenario, density) cells are re-run**. Flagged scenarios that are not in `SCN_LIST` (e.g. `i1_2p4`) are skipped and listed. Densities stay at `DENS_T` so `krig_range` is recomputed at n=327 where it is defined. If a winner lands on the *new* edge, the cell says so, and you widen again.

In [6]:
GRID_P  = (0.1, 0.25, 0.5, 1, 1.5, 2, 3, 4, 6, 8)          # was (0.5 .. 4)
GRID_NL = (3, 4, 6, 8, 12, 16, 24, 32)                       # was (4 .. 16)

def boundary_cells():
    idw, krig, skipped = set(), set(), set()
    for (nm, scn, dens, mo), v in DERIVED.items():
        if "BOUNDARY" not in str(v["source"]): continue
        if scn not in SCN_LIST: skipped.add((nm, scn, dens)); continue
        if nm.startswith("idw_power"): idw.add((scn, dens))
        if nm.startswith("krig_nlags"): krig.add((scn, dens))
    return sorted(idw), sorted(krig), sorted(skipped)

idw_c, krig_c, skipped = boundary_cells()
print(f"re-running IDW cells={len(idw_c)}, kriging cells={len(krig_c)}; skipped (scenario not in SCN_LIST): {len(skipped)}")
with timed("boundary re-run"):
    run_jobs([("derive_idw", c) for c in idw_c] + [("derive_krig", c) for c in krig_c])
    save_derived()

idw_c, krig_c, skipped2 = boundary_cells()
print("still on the (new) edge:", idw_c + krig_c or "none")
print("krig_range:", {s: round(D("krig_range", s), 3) for s in SCN_LIST})
if skipped2: print("flags left only on out-of-scope scenarios:", sorted({s[1] for s in skipped2}))

re-running IDW cells=4, kriging cells=6; skipped (scenario not in SCN_LIST): 7
--- boundary re-run ...


/home/praaneshnair/gitProjects/prism/.venv/lib/python3.12/site-packages/pykrige/ok.py:663: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 7.487616472778308e-17.
  a_inv = scipy.linalg.inv(a)
/home/praaneshnair/gitProjects/prism/.venv/lib/python3.12/site-packages/pykrige/ok.py:663: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 9.60520430027485e-18.
  a_inv = scipy.linalg.inv(a)
/home/praaneshnair/gitProjects/prism/.venv/lib/python3.12/site-packages/pykrige/ok.py:663: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 4.3234321644708404e-21.
  a_inv = scipy.linalg.inv(a)
/home/praaneshnair/gitProjects/prism/.venv/lib/python3.12/site-packages/pykrige/ok.py:663: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 9.654510431057906e-22.
  a_inv = scipy.linalg.inv(a)
/home/praaneshnair/gitProjects/prism/.venv/lib/python3.12/site-packages/pykrige/ok.py:663: LinAlgWarning: An ill-conditioned matrix detected

[parallel] job derive_idw('i2_28b', 20) done (1/10)
[parallel] job derive_idw('i2_28b', 107) done (2/10)
[parallel] job derive_idw('i3_60', 20) done (3/10)
[parallel] job derive_idw('i3_2p4', 20) done (4/10)


/home/praaneshnair/gitProjects/prism/.venv/lib/python3.12/site-packages/pykrige/ok.py:663: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 1.407828973052154e-16.
  a_inv = scipy.linalg.inv(a)
/home/praaneshnair/gitProjects/prism/.venv/lib/python3.12/site-packages/pykrige/ok.py:663: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 3.489442020189704e-21.
  a_inv = scipy.linalg.inv(a)
/home/praaneshnair/gitProjects/prism/.venv/lib/python3.12/site-packages/pykrige/ok.py:663: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 1.6055163435305439e-18.
  a_inv = scipy.linalg.inv(a)
/home/praaneshnair/gitProjects/prism/.venv/lib/python3.12/site-packages/pykrige/ok.py:663: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 1.4015635080712255e-20.
  a_inv = scipy.linalg.inv(a)
/home/praaneshnair/gitProjects/prism/.venv/lib/python3.12/site-packages/pykrige/ok.py:663: LinAlgWarning: An ill-conditioned matrix detect

[parallel] job derive_krig('i3_60', 107) done (5/10)
[parallel] job derive_krig('i1_2p5', 107) done (6/10)
[parallel] job derive_krig('i2_28b', 20) done (7/10)
[parallel] job derive_krig('i3_2p4', 20) done (8/10)
[parallel] job derive_krig('i3_2p4', 327) done (9/10)
[parallel] job derive_krig('i1_2p5', 327) done (10/10)
--- boundary re-run: 20s
still on the (new) edge: [('i2_28b', 107), ('i3_60', 20), ('i1_2p5', 107), ('i1_2p5', 327), ('i3_2p4', 20), ('i3_2p4', 327)]
krig_range: {'i2_28b': 6.46, 'i1_2p5': 3.345, 'i3_2p4': 2.689, 'i3_60': 5.039}
flags left only on out-of-scope scenarios: ['i1_2p4']


## D. `spatial_folds` and `spatial_buffer_m` (uses B and the C-updated `krig_range`)

In [7]:
with timed("derive_spatial"):
    derive_spatial(); save_derived()
print("spatial_folds =", D("spatial_folds"), "|", DERIVED[("spatial_folds", None, None, None)]["source"][:110])
print("spatial_buffer_m:", {s: round(D("spatial_buffer_m", s), 2) for s in SCN_LIST})

--- derive_spatial ...
--- derive_spatial: 6s
spatial_folds = 2 | derived: largest fold count with every block >= 327 points and >= 327 candidates beyond the variogram-range bu
spatial_buffer_m: {'i2_28b': 6.46, 'i1_2p5': 3.35, 'i3_2p4': 2.69, 'i3_60': 5.04}


## E. Missing `knn_k` searches (inner-CV GCN, k in `KS`, 5 seeds x 5 folds)

This is the most expensive step in this notebook. It runs in parallel and saves after each finished job, so an interrupt keeps progress. As in the main orchestrator, it runs *before* GNN hyperparameter tuning, so it uses the current GNN defaults. If a winner is `BOUNDARY` (k=3 or 20), widen `KS` and re-run just that cell.

In [9]:
import torch, heapq, time
torch.set_num_threads(1)

DENS_D = DENS_T
todo = [(s, n) for s in SCN_LIST for n in DENS_T if not have("knn_k", s, n)]
n_mid = DENS_T[1]

per_call = {}
for s in SCN_LIST:
    ei, ew = knn_graph(DATA[s][0], k=6)
    t = time.time()
    cv_gnn(GCN, s, n_mid, SEEDS_D[0], ei, ew)
    per_call[s] = time.time() - t
    print(f"{s}: {len(DATA[s][0]):>7} nodes, {per_call[s]:.1f}s per 5 fits")

jobs = []
for s, n in todo:
    n_calls = len([k for k in KS if k < n]) * len(SEEDS_D)   # one cv_gnn call = 5 fits
    jobs.append(per_call[s] * n_calls)
jobs.sort(reverse=True)

workers = [0.0] * max(1, N_WORKERS)
heapq.heapify(workers)
for j in jobs:
    heapq.heappush(workers, heapq.heappop(workers) + j)

print(f"pending searches: {len(todo)}")
print(f"serial total ~ {sum(jobs)/3600:.2f} h | wall on {N_WORKERS} workers ~ {max(workers)/3600:.2f} h")

i2_28b:  136151 nodes, 188.8s per 5 fits
i1_2p5:    5000 nodes, 7.5s per 5 fits
[warn] apply_normalizer: 2.9% clipped at 2 vs 1.7% on TRAIN (> +3 SE).
[warn] apply_normalizer: 2.7% clipped at 2 vs 1.9% on TRAIN (> +3 SE).
i3_2p4:   10000 nodes, 24.0s per 5 fits
i3_60:   66671 nodes, 131.6s per 5 fits
pending searches: 11
serial total ~ 9.71 h | wall on 6 workers ~ 1.84 h


In [ ]:
todo = [(s, n) for s in SCN_LIST for n in DENS_T if not have("knn_k", s, n)]
fits = sum(len([k for k in KS if k < n]) * len(SEEDS_D) * 5 for s, n in todo)
print(f"{len(todo)} searches pending (~{fits} GCN fits):", todo)
with timed("knn_k"):
    run_jobs([("derive_graph", c) for c in todo]); save_derived()
for s in SCN_LIST:
    print(s, {n: D("knn_k", s, n) for n in DENS_T if have("knn_k", s, n)})
print("BOUNDARY knn_k:", [k for k, v in DERIVED.items() if k[0] == "knn_k" and "BOUNDARY" in str(v["source"])] or "none")

11 searches pending (~1850 GCN fits): [('i2_28b', 20), ('i2_28b', 107), ('i2_28b', 327), ('i1_2p5', 107), ('i1_2p5', 327), ('i3_2p4', 20), ('i3_2p4', 107), ('i3_2p4', 327), ('i3_60', 20), ('i3_60', 107), ('i3_60', 327)]
--- knn_k ...
[warn] apply_normalizer: 0.9% clipped at 2 vs 0.8% on TRAIN (> +3 SE).


## F. (optional, default off) provisional `val_scheme`, `val_frac`, `patience`, `epochs_cap`, `grad_clip`

These are cheap-ish, but in the main orchestrator they are derived **after** the GNN hyperparameter tuning, and `get_hp` feeds into them. Run now, they would see the untuned defaults, so they are written to `provisional_rules.json` only and **never** to `derived.json`. Use them as a preview of where the values land. Cost is about 250 validation-curve fits plus about 300 for the scheme comparison.

In [ ]:
def run_provisional():
    snap, real_save = dict(DERIVED), save_derived
    globals()["save_derived"] = lambda *a, **k: None            # nothing reaches derived.json
    try:
        derive_val_scheme(); derive_training_rules(("GCN", "GAT"))
        new = {k: v for k, v in DERIVED.items() if snap.get(k) is not v}
    finally:
        globals()["save_derived"] = real_save
        DERIVED.clear(); DERIVED.update(snap)
    json.dump({_k2s(k): v for k, v in new.items() if k[0] in ("val_scheme", "val_frac", "patience", "epochs_cap", "grad_clip")},
              open("provisional_rules.json", "w"), indent=1, default=str)
    print("wrote provisional_rules.json (", len(new), "entries ); derived.json untouched")

if RUN_DEPENDENT:
    with timed("provisional rules"): run_provisional()
else:
    print("skipped (RUN_DEPENDENT = False)")

## Wrap-up

In [ ]:
DENS_D = DENS_ALL                 # restore, as the orchestrator does
save_derived()
status()
!mudslide send 919353606438 "Next: in radiomap_6g_sweep.ipynb, Run All with the SAME derived.json. The orchestrator resumes and skips everything above;"
!mudslide send 919353606438 "what remains is GCN/GAT/Hybrid/Star tuning, training rules, validation scheme, edge-weight re-check, and the seed-count pilot."